# Notebook 6: Data Insertion into MySQL

## Objective
In this notebook, we will load the cleaned and transformed PhonePe datasets into the MySQL database tables created in Notebook 5.

## Why this notebook matters
The project requires integrating extracted data into a relational SQL database before performing business-case analysis and dashboarding.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore")

In [2]:
DB_USER = "root"
DB_PASSWORD = "12345"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "phonepe_insights"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

## Load Processed CSV Files

These CSV files should have been created during the extraction and transformation phase in Notebook 4.

In [7]:
aggregated_transaction = pd.read_csv("./data/processed/aggregated_transaction.csv")
aggregated_user = pd.read_csv("./data/processed/aggregated_user.csv")
aggregated_insurance = pd.read_csv("./data/processed/aggregated_insurance.csv")

map_transaction = pd.read_csv("./data/processed/map_transaction.csv")
map_user = pd.read_csv("./data/processed/map_user.csv")
aggregated_insurance = pd.read_csv("./data/processed/aggregated_insurance.csv")

top_transaction = pd.read_csv("./data/processed/top_transaction.csv")
top_user = pd.read_csv("./data/processed/top_user.csv")


In [9]:
datasets = {
    "aggregated_transaction": aggregated_transaction,
    "aggregated_user": aggregated_user,
    "aggregated_insurance": aggregated_insurance,
    "map_transaction": map_transaction,
    "map_user": map_user,
    "aggregated_insurance": aggregated_insurance,
    "top_transaction": top_transaction,
    "top_user": top_user,
    
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

aggregated_transaction: (5034, 6)
aggregated_user: (7128, 6)
aggregated_insurance: (682, 6)
map_transaction: (20604, 6)
map_user: (20608, 6)
top_transaction: (18295, 7)
top_user: (18296, 6)


## Dataset Shape Check

This step confirms that each transformed dataset is available and ready for insertion.

In [10]:
for name, df in datasets.items():
    df.to_sql(name=name, con=engine, if_exists="append", index=False)
    print(f"{name} inserted successfully.")

aggregated_transaction inserted successfully.
aggregated_user inserted successfully.
aggregated_insurance inserted successfully.
map_transaction inserted successfully.
map_user inserted successfully.
top_transaction inserted successfully.
top_user inserted successfully.


## Data Insertion Complete

All processed data has now been pushed into the MySQL database tables.

In [11]:
for table in datasets.keys():
    query = f"SELECT COUNT(*) AS row_count FROM {table};"
    print(f"\n{table}")
    display(pd.read_sql(query, engine))


aggregated_transaction


,row_count
0,10068



aggregated_user


,row_count
0,14256



aggregated_insurance


...

,row_count
0,41216



top_transaction


,row_count
0,36590



top_user


,row_count
0,36592


## Row Count Validation

This validation ensures that data has been loaded into each table successfully.

In [12]:
null_check_queries = {
    "aggregated_transaction": """
        SELECT 
            SUM(state IS NULL) AS null_state,
            SUM(year IS NULL) AS null_year,
            SUM(quarter IS NULL) AS null_quarter
        FROM aggregated_transaction
    """,
    "map_user": """
        SELECT 
            SUM(state IS NULL) AS null_state,
            SUM(district IS NULL) AS null_district,
            SUM(registered_users IS NULL) AS null_registered_users
        FROM map_user
    """
}

for table, query in null_check_queries.items():
    print(f"\nNull check for {table}")
    display(pd.read_sql(query, engine))


Null check for aggregated_transaction


,null_state,null_year,null_quarter
0,0.0,0.0,0.0



Null check for map_user


,null_state,null_district,null_registered_users
0,0.0,0.0,0.0


In [13]:
duplicate_check_query = """
SELECT state, year, quarter, transaction_type, COUNT(*) AS duplicate_count
FROM aggregated_transaction
GROUP BY state, year, quarter, transaction_type
HAVING COUNT(*) > 1;
"""

pd.read_sql(duplicate_check_query, engine)

,state,year,quarter,transaction_type,duplicate_count
0,Andaman & Nicobar Islands,2018,1,Recharge & bill payments,2
1,Andaman & Nicobar Islands,2018,1,Peer-to-peer payments,2
2,Andaman & Nicobar Islands,2018,1,Merchant payments,2
3,Andaman & Nicobar Islands,2018,1,Financial Services,2
4,Andaman & Nicobar Islands,2018,1,Others,2
...,...,...,...,...,...
5029,West Bengal,2024,4,Merchant payments,2
5030,West Bengal,2024,4,Peer-to-peer payments,2
5031,West Bengal,2024,4,Recharge & bill payments,2
5032,West Bengal,2024,4,Financial Services,2


## Duplicate Validation

The query above checks whether duplicate records exist for the same state-year-quarter-transaction_type combination.

In [14]:
sample_query = "SELECT * FROM aggregated_transaction LIMIT 10;"
pd.read_sql(sample_query, engine)

,id,state,year,quarter,transaction_type,transaction_count,transaction_amount
0,1,Andaman & Nicobar Islands,2018,1,Recharge & bill payments,4200,1.845307e+06
1,2,Andaman & Nicobar Islands,2018,1,Peer-to-peer payments,1871,1.213866e+07
2,3,Andaman & Nicobar Islands,2018,1,Merchant payments,298,4.525072e+05
3,4,Andaman & Nicobar Islands,2018,1,Financial Services,33,1.060142e+04
4,5,Andaman & Nicobar Islands,2018,1,Others,256,1.846899e+05
5,6,Andaman & Nicobar Islands,2018,2,Recharge & bill payments,6735,2.320945e+06
6,7,Andaman & Nicobar Islands,2018,2,Peer-to-peer payments,3575,2.451193e+07
7,8,Andaman & Nicobar Islands,2018,2,Merchant payments,603,1.024491e+06
8,9,Andaman & Nicobar Islands,2018,2,Financial Services,59,1.213360e+05
9,10,Andaman & Nicobar Islands,2018,2,Others,368,3.598385e+05


## Sample Data Preview

This preview confirms that inserted values appear correctly in the relational table.

## Conclusion

In this notebook, all transformed PhonePe datasets were inserted into the MySQL database successfully. We also performed row-count, null-value, duplicate, and sample-data validations to confirm that the SQL tables are now analysis-ready.